In [ ]:
# ================================
# 1️⃣ Install dependencies
# ================================
!pip install -qU "google-genai>=1.0.0" chromadb pypdf

In [ ]:
# ================================
# 2️⃣ Imports and Gemini setup
# ================================
from pypdf import PdfReader
import chromadb
from google import genai
from google.colab import userdata

In [ ]:
# Initialize the Gemini client
GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# ================================
# 3️⃣ Load PDF from Colab directory
# ================================
pdf_path = "/content/sample_data/AI_in_modern_society.pdf"
reader = PdfReader(pdf_path)

raw_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        raw_text += text + "\n"

print("PDF loaded. Preview:")
print(raw_text[:1000])  # first 1000 chars

In [ ]:
# ================================
# 4️⃣ Chunk the document
# ================================
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text)
print("Total chunks created:", len(chunks))
print("\n--- Content of Each Chunk ---")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:\n{chunk}\n--------------------")
print("----------------------------")

In [ ]:
# ================================
# 6️⃣ Store chunks in ChromaDB
# ================================
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="simple_rag")

collection.add(
    documents=chunks,
    ids=[f"id_{i}" for i in range(len(chunks))]
)

print("Documents added to ChromaDB.")

# /root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz added to chroma db


In [ ]:
import google.generativeai as genai

# Simple RAG Query ---
while True:
    user_query = input("Ask a question (type 'exit' to quit): ")

    if user_query.lower() == 'exit':
        print("Exiting Q&A session.")
        break

    # Retrieve top 2 matches
    results = collection.query(query_texts=[user_query], n_results=2)
    context = " ".join(results['documents'][0])

    print(f"\nRetrieved chunks content:\n{results['documents'][0]}") # Added line to print chunks content

    # Generate Final Answer
    prompt = f"Context: {context}\n\nQuestion: {user_query}\n\nAnswer:"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt)

    # Correctly display the number of retrieved chunks (n_results)
    print(f"Retrieved {len(results['documents'][0])} chunks from the knowledge base.")
    print(f"\nAnswer: {response.text}")